# TEST CELLPOSE ON TRAIN SET

In [ ]:
import os
import argparse
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
from cellpose.models import CellposeModel

from metric import score
from generate_submission import build_submission
from generate_train_submission import build_submission as one_submission

In [ ]:
def get_stats(x):
    print(f"MEAN: {np.mean(x)}")
    print(f"MEDIAN: {np.median(x)}")
    print(f"MIN: {np.min(x)}")
    print(f"MAX: {np.max(x)}")

In [ ]:
def load_dax(filepath, height=2048, width=2048):
    """Load a .dax raw image file. Raw uint16 binary, no header."""
    raw = np.fromfile(filepath, dtype=np.uint16)
    n_frames = len(raw) // (height * width)
    return raw.reshape(n_frames, height, width)

In [ ]:
epi_stack = load_dax('FOV_001/Epi-750s5-635s5-545s1-473s5-408s5_001.dax')

In [ ]:
print(f'Epi stack shape: {epi_stack.shape}  (frames, height, width)')

Epi stack shape: (27, 2048, 2048)  (frames, height, width)


In [ ]:
z_plane = 2  # middle z-plane
dapi = epi_stack[6 + z_plane * 5]   # frame 16 for z2
polyt = epi_stack[5 + z_plane * 5]  # frame 15 for z2

In [ ]:
# Cellpose v4+: use CellposeModel (not models.Cellpose)
model = CellposeModel(model_type='nuclei', gpu=True)

# eval() returns 3 values: masks, flows, styles
masks, flows, styles = model.eval(dapi, diameter=30, channels=[0, 0])

print(f'Segmentation complete!')
print(f'Mask shape: {masks.shape}')
print(f'Number of cells found: {masks.max()}')

spots_train_df = pd.read_csv('spots_train.csv')
spots_train_df = spots_train_df[spots_train_df['fov'] == 'FOV_001']
score(spots_train_df)


model_type argument is not used in v4.0.1+. Ignoring this argument...
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used


Segmentation complete!
Mask shape: (2048, 2048)
Number of cells found: 218


TypeError: score() missing 2 required positional arguments: 'submission' and 'row_id_column_name'

## CALCULATE SCORE FROM 
### cell_boundaries_train.csv
### spots_train.csv


In [ ]:
from tqdm import tqdm

from shapely.geometry import Point, Polygon

poly = Polygon([(0, 0), (1, 0), (1, 1), (0, 1)])
point = Point(0.5, 0.5)
print(poly.contains(point))  # Output: True
Polygon('')

True


In [ ]:
def parse_float_list(text):
    if isinstance(text, str):
        return np.fromstring(text, sep=',').tolist()
    return None

cell_boundaries_train_df = pd.read_csv('cell_boundaries_train.csv')
cell_boundaries_train_df.iloc[:, 1:] = cell_boundaries_train_df.iloc[:, 1:].applymap(parse_float_list)

spots_train_df = pd.read_csv('spots_train.csv')
spots_train_df = spots_train_df[spots_train_df['fov'].isin(['FOV_019', 'FOV_001'])]

# For each FOV, for each Z, Mask -> split into individiual pixels
spots = spots_train_df[['fov', 'image_row', 'image_col', 'global_z']]
spots

,fov,image_row,image_col,global_z
0,FOV_019,1468,1575,0.0
1,FOV_019,303,1800,0.0
2,FOV_019,749,579,0.0
3,FOV_019,1100,704,0.0
4,FOV_019,522,118,0.0
...,...,...,...,...
1346190,FOV_001,1478,1664,4.0
1346191,FOV_001,1931,1707,4.0
1346192,FOV_001,1621,344,4.0
1346193,FOV_001,252,356,4.0


In [ ]:
cell_boundaries_train_df['']

,Unnamed: 0,boundaryX_z0,boundaryY_z0,boundaryX_z1,boundaryY_z1,boundaryX_z2,boundaryY_z2,boundaryX_z3,boundaryY_z3,boundaryX_z4,boundaryY_z4
0,267725525651232306728125526231708191,"[-1343.5731309860944, -1343.5731309860944, -13...","[-474.1219997406006, -474.230999737978, -474.3...","[-1343.5731309860944, -1343.5731309860944, -13...","[-473.9039997458458, -474.0129997432232, -474....","[-1343.6821309834718, -1343.6821309834718, -13...","[-473.03199976682663, -473.140999764204, -473....","[-1343.7911309808492, -1343.7911309808492, -13...","[-474.0129997432232, -474.1219997406006, -474....","[-1343.7911309808492, -1343.7911309808492, -13...","[-473.140999764204, -473.2499997615814, -473.3..."
1,327375991457903881984615241423065475,"[2189.4068658143283, 2189.4068658143283, 2189....","[-1787.903062850237, -1788.0120628476143, -178...","[2189.624865809083, 2189.624865809083, 2189.62...","[-1789.2110628187656, -1789.320062816143, -178...","[2189.4068658143283, 2189.4068658143283, 2189....","[-1788.6660628318787, -1788.775062829256, -178...","[2189.4068658143283, 2189.4068658143283, 2189....","[-1788.775062829256, -1788.8840628266335, -178...","[2189.624865809083, 2189.624865809083, 2189.62...","[-1788.4480628371239, -1788.5570628345013, -17..."
2,364755379140285267217928676041057312,"[-1336.3791311591863, -1336.3791311591863, -13...","[-533.744998306036, -533.8539983034134, -533.9...","[-1336.7061311513185, -1336.7061311513185, -13...","[-533.744998306036, -533.8539983034134, -533.9...","[-1334.5261312037705, -1334.5261312037705, -13...","[-531.6739983558655, -531.7829983532429, -531....","[-1333.3271312326192, -1333.338031232357, -133...","[-531.5649983584881, -531.5758983582258, -531....","[-1333.2181312352418, -1333.2181312352418, -13...","[-531.1289983689785, -531.2379983663559, -531...."
3,377956362916104527021675785082246047,"[174.90989572703836, 174.90989572703836, 174.9...","[2337.431938946247, 2337.3229389488697, 2337.2...","[174.58289573490617, 174.58289573490617, 174.5...","[2338.6309389173985, 2338.521938920021, 2338.4...","[174.255895742774, 174.255895742774, 174.25589...","[2338.3039389252663, 2338.194938927889, 2338.0...","[173.8198957532644, 173.8198957532644, 173.819...","[2337.3229389488697, 2337.2139389514923, 2337....","[173.8198957532644, 173.8198957532644, 173.819...","[2337.431938946247, 2337.3229389488697, 2337.2..."
4,429470755737170534203514668497237885,"[-1013.4271341174841, -1013.4271341174841, -10...","[-573.951002150774, -574.0600021481514, -574.1...","[-1012.010134151578, -1012.010134151578, -1012...","[-572.9700021743774, -573.0790021717548, -573....","[-1012.7731341332197, -1012.7731341332197, -10...","[-573.0790021717548, -573.1880021691322, -573....","[-1012.7731341332197, -1012.7731341332197, -10...","[-573.2970021665096, -573.406002163887, -573.5...","[-1012.7731341332197, -1012.7731341332197, -10...","[-573.8420021533966, -573.951002150774, -574.0..."
...,...,...,...,...,...,...,...,...,...,...,...
4077,339929692190473843712596262936499411592,None,None,None,None,"[-503.4611319512129, -503.4611319512129, -503....","[488.4910011589527, 488.3820011615753, 488.273...","[-503.2431319564581, -503.2431319564581, -503....","[487.61900117993355, 487.51000118255615, 487.4...","[-503.0251319617033, -503.0251319617033, -503....","[487.61900117993355, 487.51000118255615, 487.4..."
4078,340110940201655122517342329290830486652,"[1029.285869666934, 1029.285869666934, 1029.28...","[2574.4919380545616, 2574.382938057184, 2574.2...","[1028.8498696774245, 1028.8498696774245, 1028....","[2575.363938033581, 2575.2549380362034, 2575.1...","[1029.1768696695567, 1029.1768696695567, 1029....","[2579.3969379365444, 2579.287937939167, 2579.1...","[1029.285869666934, 1029.285869666934, 1029.28...","[2579.0699379444122, 2578.960937947035, 2578.8...","[1029.3948696643115, 1029.3948696643115, 1029....","[2578.74293795228, 2578.6339379549026, 2578.52..."
4079,340110949118186711251276618663953698982,None,None,"[4494.14111224710

In [ ]:
cell_boundaries_train_df.shape

(4082, 11)

In [ ]:
# This shows that some cells do not span across entire z-plane.
# cell_boundaries_train_df[cell_boundaries_train_df.iloc[:, 1:].isna().sum(axis=1) > 0]
# cell_boundaries_train_df[cell_boundaries_train_df.iloc[:, 1:].isna().sum(axis=1) == 8]
# All cells listed have at least 1 z plane
cell_boundaries_train_df[cell_boundaries_train_df.iloc[:, 1:].isna().sum(axis=1) == 10]
# cell_boundaries_train_df['Unnamed: 0'].isna().sum()

,Unnamed: 0,boundaryX_z0,boundaryY_z0,boundaryX_z1,boundaryY_z1,boundaryX_z2,boundaryY_z2,boundaryX_z3,boundaryY_z3,boundaryX_z4,boundaryY_z4


In [ ]:
spots.head()

,fov,image_row,image_col,global_z
0,FOV_019,1468,1575,0.0
1,FOV_019,303,1800,0.0
2,FOV_019,749,579,0.0
3,FOV_019,1100,704,0.0
4,FOV_019,522,118,0.0


In [ ]:
submission_df = []
# WE ITERATE ACROSS ALL SPOTS DETECTED (MASK) FIRST 
for s in tqdm(range(spots.shape[0])):
    spot_row = spots.iloc[s, :]
    z_level = int(spot_row['global_z'])
    submission_row = {
        'spot_id' : s,
        'fov' : spot_row['fov'],
        'gt_cluster_id' : 'background'
    }

    for c in tqdm(range(cell_boundaries_train_df.shape[0]), leave=False):
        cell_row = cell_boundaries_train_df.loc[c, ["Unnamed: 0", f"boundaryX_z{z_level}", f"boundaryY_z{z_level}"]]

        # print(cell_row[f"boundaryX_z{z_level}"])
        # print(cell_row)
        # print(cell_row.isna().sum() != 0)

        # if there is an NA, then the cell is not on this z-plane, skip
        if cell_row.isna().sum() != 0:
            break

        cell_x = [float(x) for x in cell_row[f"boundaryX_z{z_level}"].split(',')]
        cell_y = [float(x) for x in cell_row[f"boundaryY_z{z_level}"].split(',')]

        if ((min(cell_x) <= spot_row['image_col'] <= max(cell_x)) and 
            (min(cell_y) <= spot_row['image_row'] <= max(cell_y))):
            submission_row['gt_cluster_id'] = cell_row['"Unnamed: 0"']
            break

    submission_df.append(submission_row)

submission_df = pd.DataFrame(submission_df)

 15%|█▍        | 27765/190512 [02:29<14:37, 185.54it/s]


KeyError: '"Unnamed: 0"'

In [ ]:
submission_df.head()